<H3>PRI 2025/26: first project delivery</H3>

**GROUP X**
- name, number
- name, number
- name, number

## A) **Indexing** (preprocessing and indexing options)

### Imports

In [ ]:
import time
import sys
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

2225 2225
{'id': '001', 'category': 'business', 'text': 'Ad sales boost Time Warner profit\n\nQuarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.\n\nThe firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.\n\nTime Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL\'s underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service f

### Dataset loading

In [2]:
def load_bbc_dataset(root_dir):
    root = Path(root_dir)
    docs = []
    refs = []

    for category in ["business", "entertainment", "politics", "sport", "tech"]:
        art_dir = root / "News Articles" / category
        sum_dir = root / "Summaries" / category

        for art_file in sorted(art_dir.glob("*.txt")):
            sum_file = sum_dir / art_file.name
            if not sum_file.exists():
                continue

            with art_file.open(encoding="utf-8", errors="replace") as f:
                article_text = f.read().strip()

            with sum_file.open(encoding="utf-8", errors="replace") as f:
                summary_text = f.read().strip()

            unique_id = f"{category}_{art_file.stem}"

            docs.append({
                "id": unique_id,
                "category": category,
                "text": article_text
            })

            refs.append({
                "id": unique_id,
                "category": category,
                "summary": summary_text
            })

    return docs, refs

### Preprocessing

In [5]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

BAD_PHRASES = {"it", "which", "that", "all", "this", "these", "those"}


def normalize_phrase(phrase: str) -> str:
    """
    Preserve noun phrases as single features by replacing spaces with underscores.
    Example: 'prime minister' -> 'prime_minister'
    """
    return "_".join(phrase.lower().strip().split())


def valid_phrase(phrase: str) -> bool:
    parts = phrase.strip().split()
    return 1 <= len(parts) <= 4


def preprocess_document(text):
    """
    Preprocess one document at both document level and sentence level.

    Returns:
        {
            "tokens": [...],
            "noun_phrases": [...],
            "sentences": [...],
            "sentence_tokens": [[...], [...], ...],
            "sentence_noun_phrases": [[...], [...], ...]
        }
    """
    # Clean line breaks so title/body are less likely to merge oddly
    text = re.sub(r"\n+", " ", text).strip()

    doc = nlp(text)

    sentences = []
    sentence_tokens = []
    sentence_noun_phrases = []

    all_tokens = []
    all_noun_phrases = []

    for sent in doc.sents:
        sent_text = sent.text.strip()
        if not sent_text:
            continue

        sentences.append(sent_text)

        sent_tokens = [
            token.lemma_.lower()
            for token in sent
            if token.is_alpha and not token.is_stop and len(token.lemma_) > 1
        ]

        sent_nps = [
            normalize_phrase(chunk.text)
            for chunk in sent.noun_chunks
            if chunk.text.strip()
            and chunk.text.lower().strip() not in BAD_PHRASES
            and valid_phrase(chunk.text)
        ]

        sentence_tokens.append(sent_tokens)
        sentence_noun_phrases.append(sent_nps)

        all_tokens.extend(sent_tokens)
        all_noun_phrases.extend(sent_nps)

    return {
        "tokens": all_tokens,
        "noun_phrases": all_noun_phrases,
        "sentences": sentences,
        "sentence_tokens": sentence_tokens,
        "sentence_noun_phrases": sentence_noun_phrases,
    }

### Utility/Helper function

In [10]:
def estimate_size_bytes(obj, seen=None):
    """
    Rough recursive memory estimator for approximate in-memory size reporting.
    """
    if seen is None:
        seen = set()

    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(
            estimate_size_bytes(k, seen) + estimate_size_bytes(v, seen)
            for k, v in obj.items()
        )
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(estimate_size_bytes(x, seen) for x in obj)
    elif isinstance(obj, np.ndarray):
        size += obj.nbytes
    elif hasattr(obj, "data") and hasattr(obj, "indices") and hasattr(obj, "indptr"):
        # Approximation for scipy sparse matrices
        size += obj.data.nbytes + obj.indices.nbytes + obj.indptr.nbytes

    return size

## 1. Index builders

### 1.1 Inverted index

In [7]:
def build_inverted_index(records, id_key="id", terms_key="terms"):
    """
    Generic inverted index builder.

    records: list of dictionaries
    id_key: key for record ID
    terms_key: key for list of terms
    """
    inverted_index = defaultdict(list)
    stats = {}

    for rec in records:
        rec_id = rec[id_key]
        terms = rec[terms_key]
        counts = Counter(terms)

        for term, tf in counts.items():
            inverted_index[term].append((rec_id, tf))

        stats[rec_id] = {
            "num_tokens": len(terms),
            "num_terms": len(counts),
        }

    return dict(inverted_index), stats

### 1.2 BM25 stats

In [8]:
def compute_bm25_stats(inverted_index, stats, k1=1.2, b=0.75):
    """
    Pre-compute BM25 quantities for a collection of ranking units
    (documents or sentences).
    """
    unit_ids = list(stats.keys())
    N = len(unit_ids)

    lengths = np.array([stats[u]["num_tokens"] for u in unit_ids], dtype=float)
    avgdl = float(lengths.mean()) if N > 0 else 0.0

    idf = {}
    for term, postings in inverted_index.items():
        df = len(postings)
        idf[term] = np.log((N - df + 0.5) / (df + 0.5) + 1.0)

    return {
        "k1": k1,
        "b": b,
        "N": N,
        "avgdl": avgdl,
        "unit_ids": unit_ids,
        "unit_id_to_idx": {u: i for i, u in enumerate(unit_ids)},
        "lengths": lengths,
        "idf": idf,
    }

### 1.3 TF-IDF index

In [9]:
def build_tfidf_index(records, id_key="id", terms_key="terms"):
    """
    Build TF-IDF over pre-tokenized terms.
    Noun phrases are preserved because we join already-normalized terms.
    """
    unit_ids = []
    texts = []

    for rec in records:
        unit_ids.append(rec[id_key])
        texts.append(" ".join(rec[terms_key]))

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
    )
    matrix = vectorizer.fit_transform(texts)

    return {
        "vectorizer": vectorizer,
        "matrix": matrix,
        "unit_ids": unit_ids,
    }

### 1.4 Dense index

In [10]:
def build_dense_index(texts, unit_ids, model_name="all-MiniLM-L6-v2"):
    """
    Build dense embeddings for a list of texts.
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    return {
        "model_name": model_name,
        "unit_ids": unit_ids,
        "embeddings": embeddings,
    }

### 1.6 Main function: indexing(D, args)


In [11]:
def indexing(D, args=None):
    """
    indexing(D, args)

    Input:
        D: list of dicts with keys {"id", "category", "text"}
        args: optional configuration dictionary

    Output:
        I: hybrid index
        indexing_time: seconds
        index_size_bytes: approximate in-memory size
    """
    if args is None:
        args = {}

    start = time.time()

    use_dense = args.get("use_dense", True)
    dense_model = args.get("dense_model", "all-MiniLM-L6-v2")
    bm25_k1 = args.get("bm25_k1", 1.2)
    bm25_b = args.get("bm25_b", 0.75)

    preprocessed_docs = {}
    document_records = []
    sentence_records = []

    # 1) Preprocess all documents
    for doc in D:
        doc_id = doc["id"]
        category = doc["category"]
        text = doc["text"]

        pdata = preprocess_document(text)
        preprocessed_docs[doc_id] = pdata

        # document-level terms = tokens + noun phrases
        doc_terms = pdata["tokens"] + pdata["noun_phrases"]

        document_records.append({
            "id": doc_id,
            "doc_id": doc_id,
            "category": category,
            "text": text,
            "sentences": pdata["sentences"],
            "terms": doc_terms,
        })

        # sentence-level records
        for sent_idx, sent_text in enumerate(pdata["sentences"]):
            sent_terms = (
                pdata["sentence_tokens"][sent_idx]
                + pdata["sentence_noun_phrases"][sent_idx]
            )

            sent_id = f"{doc_id}::s{sent_idx}"

            sentence_records.append({
                "id": sent_id,
                "sent_id": sent_id,
                "doc_id": doc_id,
                "sent_idx": sent_idx,
                "category": category,
                "text": sent_text,
                "terms": sent_terms,
            })

    # 2) Document-level lexical indexes
    doc_inverted_index, doc_stats = build_inverted_index(
        document_records, id_key="id", terms_key="terms"
    )
    doc_bm25 = compute_bm25_stats(
        doc_inverted_index, doc_stats, k1=bm25_k1, b=bm25_b
    )
    doc_tfidf = build_tfidf_index(
        document_records, id_key="id", terms_key="terms"
    )

    # 3) Sentence-level lexical indexes
    sent_inverted_index, sent_stats = build_inverted_index(
        sentence_records, id_key="id", terms_key="terms"
    )
    sent_bm25 = compute_bm25_stats(
        sent_inverted_index, sent_stats, k1=bm25_k1, b=bm25_b
    )
    sent_tfidf = build_tfidf_index(
        sentence_records, id_key="id", terms_key="terms"
    )

    # 4) Dense semantic indexes
    if use_dense:
        dense_doc = build_dense_index(
            texts=[rec["text"] for rec in document_records],
            unit_ids=[rec["id"] for rec in document_records],
            model_name=dense_model,
        )

        dense_sent = build_dense_index(
            texts=[rec["text"] for rec in sentence_records],
            unit_ids=[rec["id"] for rec in sentence_records],
            model_name=dense_model,
        )
    else:
        dense_doc = None
        dense_sent = None

    # 5) Pack everything into one hybrid index
    I = {
        "documents": {
            "records": document_records,
            "preprocessed": preprocessed_docs,
            "inverted_index": doc_inverted_index,
            "stats": doc_stats,
            "bm25": doc_bm25,
            "tfidf": doc_tfidf,
            "dense": dense_doc,
        },
        "sentences": {
            "records": sentence_records,
            "inverted_index": sent_inverted_index,
            "stats": sent_stats,
            "bm25": sent_bm25,
            "tfidf": sent_tfidf,
            "dense": dense_sent,
        },
        "meta": {
            "num_docs": len(document_records),
            "num_sentences": len(sentence_records),
            "dense_model": dense_model if use_dense else None,
            "bm25_k1": bm25_k1,
            "bm25_b": bm25_b,
        }
    }

    indexing_time = time.time() - start
    index_size_bytes = estimate_size_bytes(I)

    return I, indexing_time, index_size_bytes

### 1.7 Test / sanity check

In [12]:
small_docs = docs[:100]

I_small, build_time, mem_bytes = indexing(
    small_docs,
    args={
        "use_dense": True,
        "dense_model": "all-MiniLM-L6-v2",
        "bm25_k1": 1.2,
        "bm25_b": 0.75,
    }
)

print(f"Index built in {build_time:.2f} seconds")
print(f"Approximate index size: {mem_bytes / (1024**2):.2f} MB")
print("Documents:", I_small["meta"]["num_docs"])
print("Sentences:", I_small["meta"]["num_sentences"])
print("Dense model:", I_small["meta"]["dense_model"])
print("TF-IDF doc shape:", I_small["documents"]["tfidf"]["matrix"].shape)
print("TF-IDF sent shape:", I_small["sentences"]["tfidf"]["matrix"].shape)
print("Sample doc terms:", I_small["documents"]["records"][0]["terms"][:15])
print("Sample sent terms:", I_small["sentences"]["records"][0]["terms"][:15])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2789.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2974.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 45/45 [00:16<00:00,  2.74it/s]


Index built in 45.89 seconds
Approximate index size: 13.28 MB
Documents: 100
Sentences: 1435
Dense model: all-MiniLM-L6-v2
TF-IDF doc shape: (100, 7770)
TF-IDF sent shape: (1435, 7770)
Sample doc terms: ['ad', 'sale', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profit', 'medium', 'giant', 'timewarner', 'jump', 'month', 'december', 'year']
Sample sent terms: ['ad', 'sale', 'boost', 'time', 'warner', 'profit', 'quarterly', 'profit', 'medium', 'giant', 'timewarner', 'jump', 'month', 'december', 'year']


In [13]:
docs, refs = load_bbc_dataset("data/BBC News Summary")

I, build_time, mem_bytes = indexing(
    docs,
    args={
        "use_dense": True,
        "dense_model": "all-MiniLM-L6-v2",
        "bm25_k1": 1.2,
        "bm25_b": 0.75,
    }
)

print(f"Index built in {build_time:.2f} seconds")
print(f"Approximate index size: {mem_bytes / (1024**2):.2f} MB")
print("Documents:", I["meta"]["num_docs"])
print("Sentences:", I["meta"]["num_sentences"])
print("Dense model:", I["meta"]["dense_model"])

print("\nSample document record:")
print(I["documents"]["records"][0])

print("\nSample sentence record:")
print(I["sentences"]["records"][0])

KeyboardInterrupt: 

**We implemented a hybrid index with lexical and semantic components at both document and sentence levels.
The lexical side includes inverted indexes, TF-IDF vector spaces, and BM25 statistics.
The semantic side includes dense embeddings computed with a lightweight encoder.
To align with the project statement, the indexed lexical units include both lemmatized words and normalized noun phrases, where multiword noun phrases are preserved as single features using underscore concatenation.
Document-level structures support keyword extraction, while sentence-level structures support extractive summarization and later MMR-based reranking.**

## B) **Extractive summarization**

### *B.1 Summarization baseline solution: results for a given document*

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np

In [15]:
# cache dense models so they are not reloaded every call
_dense_model_cache = {}

def get_dense_model(model_name):
    if model_name not in _dense_model_cache:
        _dense_model_cache[model_name] = SentenceTransformer(model_name)
    return _dense_model_cache[model_name]

In [16]:
def score_sentences(sentences, sentence_terms, doc_text, doc_terms, I, method="tfidf", args=None):
    if args is None:
        args = {}

    if method == "tfidf":
        vectorizer = I["sentences"]["tfidf"]["vectorizer"]

        sent_texts = [" ".join(terms) for terms in sentence_terms]
        doc_representation = " ".join(doc_terms)

        sent_vectors = vectorizer.transform(sent_texts)
        doc_vector = vectorizer.transform([doc_representation])

        scores = cosine_similarity(sent_vectors, doc_vector).flatten()
        vectors_for_mmr = sent_vectors.toarray()

        return scores, vectors_for_mmr

    elif method == "bm25":
        bm25 = BM25Okapi(sentence_terms)
        query_terms = doc_terms

        scores = np.array(bm25.get_scores(query_terms), dtype=float)

        vectorizer = I["sentences"]["tfidf"]["vectorizer"]
        sent_texts = [" ".join(terms) for terms in sentence_terms]
        vectors_for_mmr = vectorizer.transform(sent_texts).toarray()

        return scores, vectors_for_mmr

    elif method == "embedding":
        model_name = args.get("dense_model", I["meta"]["dense_model"])
        encoder = get_dense_model(model_name)

        sent_embeddings = encoder.encode(sentences, convert_to_numpy=True)
        doc_embedding = encoder.encode([doc_text], convert_to_numpy=True)

        scores = cosine_similarity(sent_embeddings, doc_embedding).flatten()
        vectors_for_mmr = sent_embeddings

        return scores, vectors_for_mmr

    else:
        raise ValueError("Unknown summarization method")

In [17]:
def mmr_rerank(sentences, scores, vectors, p, lambda_param=0.5):
    selected = []
    candidates = list(range(len(sentences)))

    while len(selected) < p and candidates:
        mmr_scores = []

        for i in candidates:
            relevance = scores[i]

            redundancy = 0.0
            if selected:
                redundancy = max(
                    cosine_similarity(
                        vectors[i].reshape(1, -1),
                        vectors[j].reshape(1, -1)
                    )[0][0]
                    for j in selected
                )

            mmr_score = (1 - lambda_param) * relevance - lambda_param * redundancy
            mmr_scores.append((i, mmr_score))

        best_idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(best_idx)
        candidates.remove(best_idx)

    return selected

In [18]:
def summarization(d, p=3, l=None, I=None, args=None):
    if args is None:
        args = {"method": "tfidf"}

    if I is None:
        raise ValueError("I must be provided")

    method = args.get("method", "tfidf")
    use_mmr = args.get("use_mmr", False)
    lambda_param = args.get("lambda_param", 0.5)

    pdata = preprocess_document(d)
    sentences = pdata["sentences"]

    if len(sentences) == 0:
        return []

    sentence_terms = [
        pdata["sentence_tokens"][i] + pdata["sentence_noun_phrases"][i]
        for i in range(len(sentences))
    ]
    doc_terms = pdata["tokens"] + pdata["noun_phrases"]

    if p > len(sentences):
        p = len(sentences)

    scores, vectors_for_mmr = score_sentences(
        sentences=sentences,
        sentence_terms=sentence_terms,
        doc_text=d,
        doc_terms=doc_terms,
        I=I,
        method=method,
        args=args
    )

    if use_mmr:
        ranked_indices = mmr_rerank(
            sentences=sentences,
            scores=scores,
            vectors=vectors_for_mmr,
            p=p,
            lambda_param=lambda_param
        )
    else:
        ranked_indices = list(np.argsort(scores)[::-1][:p])

    result = [(int(i), float(scores[i])) for i in ranked_indices]

    if l is not None:
        limited = []
        total_chars = 0
        for i, score in result:
            sent_len = len(sentences[i])
            if total_chars + sent_len <= l or not limited:
                limited.append((i, score))
                total_chars += sent_len
            else:
                break
        result = limited

    return result

In [11]:
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

summary = summarization(
    doc,
    p=3,
    I=I_small,
    args={"method": "tfidf"}
)

print(summary)
for i, score in summary:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

[(15, 0.48840104930884), (2, 0.4701582350500295), (0, 0.46812487742904924)]
[15] 0.4884 -> TimeWarner is to restate its accounts as part of efforts to resolve an inquiry into AOL by US market regulators.
[2] 0.4702 -> TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn.
[0] 0.4681 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.


### B.2 IR models (TF-IDF, BM25, lightweight encoder LLMs)

In [20]:
def score_sentences(sentences, sentence_terms, doc_text, doc_terms, I, method="tfidf", args=None):
    """
    Score document sentences against the document using one IR model.

    Returns:
        scores: np.ndarray of shape (num_sentences,)
        vectors_for_mmr: dense vectors used later for MMR redundancy
    """
    if args is None:
        args = {}

    # ----- TF-IDF -----
    if method == "tfidf":
        vectorizer = I["sentences"]["tfidf"]["vectorizer"]

        sent_texts = [" ".join(terms) for terms in sentence_terms]
        doc_representation = " ".join(doc_terms)

        sent_vectors = vectorizer.transform(sent_texts)
        doc_vector = vectorizer.transform([doc_representation])

        scores = cosine_similarity(sent_vectors, doc_vector).flatten()
        vectors_for_mmr = sent_vectors.toarray()

        return scores, vectors_for_mmr

    # ----- BM25 -----
    elif method == "bm25":
        bm25 = BM25Okapi(sentence_terms)
        query_terms = doc_terms

        scores = np.array(bm25.get_scores(query_terms), dtype=float)

        # use TF-IDF sentence vectors for redundancy estimation in MMR
        vectorizer = I["sentences"]["tfidf"]["vectorizer"]
        sent_texts = [" ".join(terms) for terms in sentence_terms]
        vectors_for_mmr = vectorizer.transform(sent_texts).toarray()

        return scores, vectors_for_mmr

    # ----- Dense encoder LLM -----
    elif method == "embedding":
        model_name = args.get("dense_model", I["meta"]["dense_model"])
        encoder = get_dense_model(model_name)

        sent_embeddings = encoder.encode(sentences, convert_to_numpy=True)
        doc_embedding = encoder.encode([doc_text], convert_to_numpy=True)

        scores = cosine_similarity(sent_embeddings, doc_embedding).flatten()
        vectors_for_mmr = sent_embeddings

        return scores, vectors_for_mmr

    else:
        raise ValueError("Unknown summarization method")

In [21]:
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

for method in ["tfidf", "bm25", "embedding"]:
    print("\nMODEL:", method)

    summary = summarization(
        doc,
        p=3,
        I=I_small,
        args={"method": method}
    )

    for i, score in summary:
        print(f"[{i}] {score:.4f} -> {sentences[i]}")


MODEL: tfidf
[15] 0.4884 -> TimeWarner is to restate its accounts as part of efforts to resolve an inquiry into AOL by US market regulators.
[2] 0.4702 -> TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn.
[0] 0.4681 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

MODEL: bm25
[17] 75.9760 -> The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
[11] 68.1123 -> But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.
[8] 66.2084 -> It ho

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4384.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.
[7] 0.6572 -> However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.


In [22]:
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

for model_name in ["all-MiniLM-L6-v2", "multi-qa-MiniLM-L6-cos-v1"]:
    print("\nDENSE MODEL:", model_name)

    summary = summarization(
        doc,
        p=3,
        I=I_small,
        args={
            "method": "embedding",
            "dense_model": model_name
        }
    )

    for i, score in summary:
        print(f"[{i}] {score:.4f} -> {sentences[i]}")


DENSE MODEL: all-MiniLM-L6-v2
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.
[7] 0.6572 -> However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.

DENSE MODEL: multi-qa-MiniLM-L6-cos-v1


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7220.20it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0] 0.8699 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[12] 0.7372 -> For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.
[10] 0.7295 -> Time Warner's fourth quarter profits were slightly better than analysts' expectations.


### B.4 Maximal Marginal Relevance

In [23]:
#code, statistics and/or charts here
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

print("WITHOUT MMR")
summary_no_mmr = summarization(
    doc,
    p=3,
    I=I_small,
    args={
        "method": "embedding",
        "dense_model": "all-MiniLM-L6-v2",
        "use_mmr": False
    }
)

for i, score in summary_no_mmr:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

print("\nWITH MMR (lambda=0.5)")
summary_mmr = summarization(
    doc,
    p=3,
    I=I_small,
    args={
        "method": "embedding",
        "dense_model": "all-MiniLM-L6-v2",
        "use_mmr": True,
        "lambda_param": 0.5
    }
)

for i, score in summary_mmr:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

WITHOUT MMR
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.
[7] 0.6572 -> However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.

WITH MMR (lambda=0.5)
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[5] 0.5533 -> But its own internet business, AOL, had has mixed fortunes.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.


In [24]:
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"\nMMR with lambda = {lam}")

    summary = summarization(
        doc,
        p=3,
        I=I_small,
        args={
            "method": "embedding",
            "dense_model": "all-MiniLM-L6-v2",
            "use_mmr": True,
            "lambda_param": lam
        }
    )

    for i, score in summary:
        print(f"[{i}] {score:.4f} -> {sentences[i]}")


MMR with lambda = 0.0
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.
[7] 0.6572 -> However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.

MMR with lambda = 0.25
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now owns 8% of search-engine Google.
[8] 0.6510 -> It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for high-speed broadband.

MMR with lambda = 0.5
[0] 0.8281 -> Ad sales boost Time Warner profit

## C) **Abstractive summarization**

### *C.1 Prompting decoder LLMs*

##### C.1.1 Imports og cache

In [ ]:
#code, statistics and/or charts here
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

[(17, 0.680005794257016), (12, 0.6306804450999088), (11, 0.6219844445847503)]
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue. 0.680005794257016
For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn. 0.6306804450999088
But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results. 0.6219844445847503


In [26]:
_decoder_cache = {}

def get_decoder_model(model_name):
    """
    Cache decoder LLMs so they are not reloaded every call.
    """
    if model_name not in _decoder_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        _decoder_cache[model_name] = (tokenizer, model)

    return _decoder_cache[model_name]

##### C.1.2 Prompt builder

In [ ]:
def build_summary_prompt(document_text, prompt_type="focused", max_sentences=3):
    if prompt_type == "focused":
        return (
            f"Read the news article below.\n"
            f"Write at most {max_sentences} factual sentences summarizing only the main events.\n"
            f"Use only information from the article.\n"
            f"Do not invent names, companies, dates, or numbers.\n"
            f"Do not add opinions.\n\n"
            f"ARTICLE:\n{document_text}\n\n"
            f"SUMMARY:\n"
        )

    elif prompt_type == "brief":
        return (
            f"Read the article below.\n"
            f"Write exactly {max_sentences} short summary sentences.\n"
            f"Each sentence must mention one important fact from the article.\n"
            f"Do not invent details.\n\n"
            f"ARTICLE:\n{document_text}\n\n"
            f"NEWS BRIEF:\n"
        )

    else:
        raise ValueError("Unknown prompt_type")


MODEL: tfidf
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.
But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.

MODEL: bm25
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
But its film

##### C.1.3 Abstractive summarization

In [28]:
def abstractive_summarization(d, p=3, l=None, I=None, args=None):
    """
    Abstractive summarization of a single document using a decoder LLM.

    Input:
        d    : raw document text
        p    : max number of sentences
        l    : optional max number of characters
        I    : kept for interface consistency
        args : {
            "model_name": ...,
            "prompt_type": ...,
            "max_new_tokens": ...
        }

    Output:
        generated summary text
    """
    if args is None:
        args = {}

    model_name = args.get("model_name", "distilgpt2")
    prompt_type = args.get("prompt_type", "focused")
    max_new_tokens = args.get("max_new_tokens", 100)

    tokenizer, model = get_decoder_model(model_name)

    # trim long docs a bit for small decoder models
    clean_doc = re.sub(r"\s+", " ", d).strip()
    clean_doc = clean_doc[:1500]

    prompt = build_summary_prompt(
        document_text=clean_doc,
        prompt_type=prompt_type,
        max_sentences=p
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
)

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "SUMMARY:" in full_text:
        summary = full_text.split("SUMMARY:", 1)[-1].strip()
    elif "NEWS BRIEF:" in full_text:
        summary = full_text.split("NEWS BRIEF:", 1)[-1].strip()
    else:
        summary = full_text.strip()

    if l is not None:
        summary = summary[:l].strip()

    return summary

Method: TF-IDF + MMR (λ = 0.5)
--------------------------------------------------------------------------------
[17] score = 0.6800
The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.

[11] score = 0.6220
But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.

[12] score = 0.6307
For the full-year, TimeWarner posted a profit of $3.36bn, up 27% from its 2003 performance, while revenues grew 6.4% to $42.09bn.


Method: Embeddings (MiniLM) + MMR (λ = 0.5)
--------------------------------------------------------------------------------
[00] score = 0.8281
Ad sales boost Time Warn

##### C.1.4 Test one document

In [29]:
doc = docs[0]["text"]

summary = abstractive_summarization(
    doc,
    p=3,
    I=I_small,
    args={
    "model_name": "gpt2",
    "prompt_type": "focused",
    "max_new_tokens": 60
}
)

print(summary)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3529.78it/s]


The top 10 ad buyers are as follows...


##### C.1.5 Compare two prompts

In [30]:
doc = docs[0]["text"]

for prompt_type in ["focused", "brief"]:
    print("\nPROMPT TYPE:", prompt_type)

    summary = abstractive_summarization(
        doc,
        p=3,
        I=I_small,
        args={
            "model_name": "distilgpt2",
            "prompt_type": prompt_type,
            "max_new_tokens": 100
        }
    )

    print(summary)


PROMPT TYPE: focused


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2870.04it/s]


Time Warner International shares increased 16% over the period ending November 31, 2013. All previous quarterly earnings are based upon an adjusted gross domestic product excluding foreign currency units sold during comparable periods between January 1, 2014 and May 30, 2015. This includes all consolidated stock trading including dividends paid as part of acquisitions into individual equity funds through other publicly traded securities exchange transactions conducted with Wells Fargo Inc. In addition - such purchases occur once per month across multiple public markets within the United States. *
Company share price

PROMPT TYPE: brief
The latest analysis shows Amazon may be holding off buying much of its Kindle Fire tablet market as well as an acquisition deal with Alphabet Inc., owned by Yahoo Group Holding Ltd., another major investor.


##### C.1.6 Compare two decoder models

In [31]:
doc = docs[0]["text"]

for model_name in ["distilgpt2", "gpt2"]:
    print("\nMODEL:", model_name)

    summary = abstractive_summarization(
        doc,
        p=3,
        I=I_small,
        args={
            "model_name": model_name,
            "prompt_type": "focused",
            "max_new_tokens": 60
        }
    )

    print(summary)


MODEL: distilgpt2
News story about Amazon’s acquisition of Hulu was first reported earlier this month after reports emerged last week that Sony Pictures Entertainment would pay £14.4bn to buy Netflix Inc., but some sources say they have been unable to find any evidence supporting an agreement between Paramount Television Corp Ltd. and Universal

MODEL: gpt2
"It was disappointing," says Tim O'Brien, chief executive of digital strategy consultancy C&C International Group . "The first five days are interesting because we have seen some very big stories." On Sunday , his team laid out their plan behind selling 1 million shares worth around £20billion ($30


### C.2 Reciprocal Rank Fusion

In [32]:
#code, statistics and/or charts here
def get_sentence_rankings(d, I, method="tfidf", args=None):
    """
    Return full ranked list of sentence indices for one document.
    """
    if args is None:
        args = {}

    pdata = preprocess_document(d)
    sentences = pdata["sentences"]

    if len(sentences) == 0:
        return [], np.array([]), []

    sentence_terms = [
        pdata["sentence_tokens"][i] + pdata["sentence_noun_phrases"][i]
        for i in range(len(sentences))
    ]
    doc_terms = pdata["tokens"] + pdata["noun_phrases"]

    scores, _ = score_sentences(
        sentences=sentences,
        sentence_terms=sentence_terms,
        doc_text=d,
        doc_terms=doc_terms,
        I=I,
        method=method,
        args=args
    )

    ranked_indices = list(np.argsort(scores)[::-1])
    return ranked_indices, scores, sentences

In [33]:
# RRF function
def reciprocal_rank_fusion(rank_lists, mu=5):
    """
    Fuse multiple ranked lists with Reciprocal Rank Fusion.

    rank_lists: list of ranked lists, each containing sentence indices
    mu: fusion constant

    Returns:
        fused_ranking: list of sentence indices sorted by RRF score
        fused_scores: dict sentence_idx -> RRF score
    """
    fused_scores = {}

    for rank_list in rank_lists:
        for rank_pos, sent_idx in enumerate(rank_list, start=1):
            fused_scores[sent_idx] = fused_scores.get(sent_idx, 0.0) + 1.0 / (mu + rank_pos)

    fused_ranking = sorted(
        fused_scores.keys(),
        key=lambda x: fused_scores[x],
        reverse=True
    )

    return fused_ranking, fused_scores

In [34]:
# RRF summarization
def rrf_summarization(d, p=3, l=None, I=None, args=None):
    """
    Extractive summarization using RRF over sparse and dense sentence rankings.

    args:
        {
            "sparse_method": "bm25" or "tfidf",
            "dense_model": "...",
            "mu": 5
        }

    Output:
        list of (sentence_position, rrf_score)
    """
    if args is None:
        args = {}

    if I is None:
        raise ValueError("I must be provided")

    sparse_method = args.get("sparse_method", "bm25")
    dense_model = args.get("dense_model", I["meta"]["dense_model"])
    mu = args.get("mu", 5)

    pdata = preprocess_document(d)
    sentences = pdata["sentences"]

    if len(sentences) == 0:
        return []

    sparse_rank, sparse_scores, _ = get_sentence_rankings(
        d,
        I,
        method=sparse_method,
        args={}
    )

    dense_rank, dense_scores, _ = get_sentence_rankings(
        d,
        I,
        method="embedding",
        args={"dense_model": dense_model}
    )

    fused_rank, fused_scores = reciprocal_rank_fusion(
        [sparse_rank, dense_rank],
        mu=mu
    )

    selected = fused_rank[:p]
    result = [(int(i), float(fused_scores[i])) for i in selected]

    if l is not None:
        limited = []
        total_chars = 0

        for i, score in result:
            sent_len = len(sentences[i])
            if total_chars + sent_len <= l or not limited:
                limited.append((i, score))
                total_chars += sent_len
            else:
                break

        result = limited

    return result

In [35]:
# test
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

print("BM25")
summary_bm25 = summarization(
    doc,
    p=3,
    I=I_small,
    args={"method": "bm25"}
)
for i, score in summary_bm25:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

print("\nDENSE")
summary_dense = summarization(
    doc,
    p=3,
    I=I_small,
    args={
        "method": "embedding",
        "dense_model": "all-MiniLM-L6-v2"
    }
)
for i, score in summary_dense:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

print("\nRRF (BM25 + DENSE)")
summary_rrf = rrf_summarization(
    doc,
    p=3,
    I=I_small,
    args={
        "sparse_method": "bm25",
        "dense_model": "all-MiniLM-L6-v2",
        "mu": 5
    }
)
for i, score in summary_rrf:
    print(f"[{i}] {score:.4f} -> {sentences[i]}")

BM25
[17] 75.9760 -> The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal with German music publisher Bertelsmann's purchase of a stake in AOL Europe, which it had reported as advertising revenue.
[11] 68.1123 -> But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.
[8] 66.2084 -> It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for high-speed broadband.

DENSE
[0] 0.8281 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[4] 0.6702 -> Time Warner said on Friday that it now ow

In [36]:
# 2 dense encoders in RRF
doc = docs[0]["text"]
sentences = preprocess_document(doc)["sentences"]

for dense_model in ["all-MiniLM-L6-v2", "multi-qa-MiniLM-L6-cos-v1"]:
    print("\nRRF with dense model:", dense_model)

    summary_rrf = rrf_summarization(
        doc,
        p=3,
        I=I_small,
        args={
            "sparse_method": "bm25",
            "dense_model": dense_model,
            "mu": 5
        }
    )

    for i, score in summary_rrf:
        print(f"[{i}] {score:.4f} -> {sentences[i]}")


RRF with dense model: all-MiniLM-L6-v2
[0] 0.2778 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[8] 0.2361 -> It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for high-speed broadband.
[7] 0.2250 -> However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues.

RRF with dense model: multi-qa-MiniLM-L6-cos-v1
[0] 0.2778 -> Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.
[17] 0.2222 -> The company said it was unable to estimate the amount it needed to set aside for legal reserves, which it previously set at $500m. It intends to adjust the way it accounts for a deal wi

## D) **Keyword extraction**

#### D.1 Extractive keyword extraction

In [ ]:
#code, statistics and/or charts here
from collections import Counter
import math
import re

In [38]:

def clean_keyword(kw):
    kw = kw.strip().lower()
    kw = re.sub(r"_+", "_", kw)
    return kw


BAD_KEYPHRASES = {
    "it", "which", "that", "this", "these", "those", "all",
    "he", "she", "they", "we", "i"
}

BAD_FIRST_WORDS = {"a", "an", "the", "this", "that", "these", "those"}


def valid_keyword_candidate(kw):
    if not kw:
        return False
    if kw in BAD_KEYPHRASES:
        return False

    parts = kw.split("_")

    if len(parts) > 4:
        return False

    if parts[0] in BAD_FIRST_WORDS:
        return False

    if all(len(part) <= 1 for part in parts):
        return False

    return True

In [39]:
def score_keyword_candidates_tfidf(candidates, I):
    """
    Score keyword candidates using simple TF-IDF:
        score = tf(candidate, d) * idf(candidate, D)
    """
    if len(candidates) == 0:
        return {}

    candidate_counts = Counter(candidates)

    inverted_index = I["documents"]["inverted_index"]
    N = I["documents"]["bm25"]["N"]

    scores = {}

    for cand, tf in candidate_counts.items():
        df = len(inverted_index.get(cand, []))
        if df == 0:
            continue

        # smooth IDF
        idf = math.log((N + 1) / (df + 1)) + 1.0
        scores[cand] = tf * idf

    return scores

In [40]:
def keyword_extraction(d, p=5, I=None, args=None):
    """
    Extract top-p informative keywords for one document.

    Supports:
        - extractive TF-IDF-based keyword extraction
        - generative keyword extraction (optional)

    Output:
        ordered list of keywords
    """
    if args is None:
        args = {"method": "tfidf"}

    if I is None:
        raise ValueError("I must be provided")

    method = args.get("method", "tfidf")

    pdata = preprocess_document(d)

    # ----- Extractive TF-IDF keywords -----
    if method == "tfidf":
        # primarily noun phrases, as requested in the project
        candidates = [clean_keyword(np_) for np_ in pdata["noun_phrases"]]
        candidates = [c for c in candidates if valid_keyword_candidate(c)]

        # fallback: if too few noun phrases, add single tokens
        if len(set(candidates)) < p:
            token_candidates = [
                clean_keyword(tok)
                for tok in pdata["tokens"]
                if valid_keyword_candidate(tok)
            ]
            candidates = candidates + token_candidates

        scores = score_keyword_candidates_tfidf(candidates, I)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        # deduplicate while preserving order
        seen = set()
        keywords = []
        for kw, score in ranked:
            if kw not in seen:
                seen.add(kw)
                keywords.append(kw)
            if len(keywords) == p:
                break

        return keywords

    # ----- Generative keyword extraction -----
    elif method == "generative":
        return generative_keyword_extraction(d, p=p, args=args)

    else:
        raise ValueError("Unknown keyword extraction method")

In [41]:
doc = docs[0]["text"]

keywords_tfidf = keyword_extraction(
    doc,
    p=5,
    I=I_small,
    args={"method": "tfidf"}
)

print("TF-IDF keywords:")
print(keywords_tfidf)

TF-IDF keywords:
['timewarner', 'aol', 'aol_europe', 'ad_sales', 'us_media_giant']


#### D.2 Generative keyword extraction

In [42]:
def build_keyword_prompt(document_text, p=5, prompt_type="phrases"):
    if prompt_type == "phrases":
        return (
            f"Read the news article below.\n"
            f"Return exactly {p} informative keywords or short keyphrases.\n"
            f"Use only information from the article.\n"
            f"Prefer noun phrases.\n"
            f"Return only a comma-separated list.\n\n"
            f"ARTICLE:\n{document_text}\n\n"
            f"KEYWORDS:"
        )

    elif prompt_type == "compact":
        return (
            f"Extract exactly {p} short indexing keyphrases from the article below.\n"
            f"Do not explain anything.\n"
            f"Return only comma-separated keyphrases.\n\n"
            f"ARTICLE:\n{document_text}\n\n"
            f"KEYWORDS:"
        )

    else:
        raise ValueError("Unknown prompt_type")

In [43]:
def generative_keyword_extraction(d, p=5, args=None):
    if args is None:
        args = {}

    model_name = args.get("model_name", "gpt2")
    prompt_type = args.get("prompt_type", "phrases")
    max_new_tokens = args.get("max_new_tokens", 40)

    tokenizer, model = get_decoder_model(model_name)
    model.eval()

    clean_doc = re.sub(r"\s+", " ", d).strip()
    clean_doc = clean_doc[:1200]

    prompt = build_keyword_prompt(
        document_text=clean_doc,
        p=p,
        prompt_type=prompt_type
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "KEYWORDS:" in full_text:
        kw_text = full_text.split("KEYWORDS:", 1)[-1].strip()
    else:
        kw_text = full_text.strip()

    # simple parsing
    raw_keywords = [x.strip().lower() for x in kw_text.split(",")]
    raw_keywords = [clean_keyword(x.replace(" ", "_")) for x in raw_keywords]
    raw_keywords = [x for x in raw_keywords if valid_keyword_candidate(x)]

    # deduplicate
    seen = set()
    keywords = []
    for kw in raw_keywords:
        if kw not in seen:
            seen.add(kw)
            keywords.append(kw)
        if len(keywords) == p:
            break

    return keywords

In [44]:
doc = docs[0]["text"]

print("TF-IDF keywords")
print(
    keyword_extraction(
        doc,
        p=5,
        I=I_small,
        args={"method": "tfidf"}
    )
)

print("\nGenerative keywords (gpt2, prompt=phrases)")
print(
    keyword_extraction(
        doc,
        p=5,
        I=I_small,
        args={
            "method": "generative",
            "model_name": "gpt2",
            "prompt_type": "phrases",
            "max_new_tokens": 40
        }
    )
)

print("\nGenerative keywords (gpt2, prompt=compact)")
print(
    keyword_extraction(
        doc,
        p=5,
        I=I_small,
        args={
            "method": "generative",
            "model_name": "gpt2",
            "prompt_type": "compact",
            "max_new_tokens": 40
        }
    )
)

TF-IDF keywords
['timewarner', 'aol', 'aol_europe', 'ad_sales', 'us_media_giant']

Generative keywords (gpt2, prompt=phrases)
[]

Generative keywords (gpt2, prompt=compact)
['ad', 'news', 'ads', 'marketing', 'web_sites']


## E) **Evaluation**

#### E.1 Evaluation setup

In [45]:
#code, statistics and/or charts here
import re
import numpy as np
from collections import defaultdict

In [46]:


def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def sentences_match(pred_sentence, ref_sentence, threshold=0.6):
    pred_sentence = normalize_text(pred_sentence)
    ref_sentence = normalize_text(ref_sentence)

    if pred_sentence == ref_sentence:
        return True

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform([pred_sentence, ref_sentence])
    sim = cosine_similarity(X[0], X[1])[0][0]

    return sim >= threshold

In [47]:
def get_reference_sentences(ref_text):
    ref_pdata = preprocess_document(ref_text)
    return [normalize_text(s) for s in ref_pdata["sentences"] if s.strip()]

In [48]:
ref_lookup = {r["id"]: r for r in refs}
doc_lookup = {d["id"]: d for d in docs}

#### E.2 Extractive evaluation metrics

In [49]:
def extract_selected_sentences(doc_text, summary_output):
    """
    summary_output: list of (sentence_idx, score)
    """
    pdata = preprocess_document(doc_text)
    sentences = pdata["sentences"]

    selected = []
    for idx, score in summary_output:
        if 0 <= idx < len(sentences):
            selected.append(normalize_text(sentences[idx]))
    return selected

In [50]:
def precision_recall_fbeta(predicted_sentences, reference_sentences, beta=0.5, threshold=0.6):
    pred_list = list(dict.fromkeys(predicted_sentences))
    ref_list = list(dict.fromkeys(reference_sentences))

    matched_pred = 0
    for pred in pred_list:
        if any(sentences_match(pred, ref, threshold=threshold) for ref in ref_list):
            matched_pred += 1

    precision = matched_pred / len(pred_list) if pred_list else 0.0

    matched_ref = 0
    for ref in ref_list:
        if any(sentences_match(pred, ref, threshold=threshold) for pred in pred_list):
            matched_ref += 1

    recall = matched_ref / len(ref_list) if ref_list else 0.0

    if precision == 0 and recall == 0:
        fbeta = 0.0
    else:
        beta2 = beta ** 2
        fbeta = (1 + beta2) * precision * recall / (beta2 * precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "fbeta": fbeta
    }

In [51]:
def average_precision_from_ranking(doc_text, ranked_output, reference_sentences, threshold=0.6):
    pdata = preprocess_document(doc_text)
    sentences = [normalize_text(s) for s in pdata["sentences"]]

    hits = 0
    precision_sum = 0.0
    already_matched_refs = set()

    for rank, (idx, score) in enumerate(ranked_output, start=1):
        if 0 <= idx < len(sentences):
            sent = sentences[idx]

            matched_ref_idx = None
            for j, ref in enumerate(reference_sentences):
                if j not in already_matched_refs and sentences_match(sent, ref, threshold=threshold):
                    matched_ref_idx = j
                    break

            if matched_ref_idx is not None:
                already_matched_refs.add(matched_ref_idx)
                hits += 1
                precision_sum += hits / rank

    return precision_sum / len(reference_sentences) if reference_sentences else 0.0

In [52]:
def full_ranked_summarization(d, I=None, args=None):
    """
    Return full ranked extractive output [(sentence_idx, score), ...]
    without truncating to p.
    """
    if args is None:
        args = {"method": "tfidf"}

    pdata = preprocess_document(d)
    sentences = pdata["sentences"]

    if len(sentences) == 0:
        return []

    sentence_terms = [
        pdata["sentence_tokens"][i] + pdata["sentence_noun_phrases"][i]
        for i in range(len(sentences))
    ]
    doc_terms = pdata["tokens"] + pdata["noun_phrases"]

    scores, _ = score_sentences(
        sentences=sentences,
        sentence_terms=sentence_terms,
        doc_text=d,
        doc_terms=doc_terms,
        I=I,
        method=args.get("method", "tfidf"),
        args=args
    )

    ranked_indices = list(np.argsort(scores)[::-1])
    return [(int(i), float(scores[i])) for i in ranked_indices]

#### E.3 Evaluate one document / one method

In [53]:
def evaluate_one_document(doc_item, ref_item, I, method="tfidf", beta=0.5, p=3, args=None, threshold=0.6):
    if args is None:
        args = {}

    doc_text = doc_item["text"]
    ref_text = ref_item["summary"]

    summary_output = summarization(
        doc_text,
        p=p,
        I=I,
        args={"method": method, **args}
    )

    predicted_sentences = extract_selected_sentences(doc_text, summary_output)
    reference_sentences = get_reference_sentences(ref_text)

    prf = precision_recall_fbeta(
        predicted_sentences=predicted_sentences,
        reference_sentences=reference_sentences,
        beta=beta,
        threshold=threshold
    )

    ranked_output = full_ranked_summarization(
        doc_text,
        I=I,
        args={"method": method, **args}
    )

    ap = average_precision_from_ranking(
        doc_text=doc_text,
        ranked_output=ranked_output,
        reference_sentences=reference_sentences,
        threshold=threshold
    )

    return {
        "doc_id": doc_item["id"],
        "category": doc_item["category"],
        "method": method,
        "precision": prf["precision"],
        "recall": prf["recall"],
        "fbeta": prf["fbeta"],
        "average_precision": ap,
        "predicted_sentences": predicted_sentences,
        "reference_sentences": reference_sentences
    }

In [54]:
doc_item = docs[0]
ref_item = ref_lookup[doc_item["id"]]

result = evaluate_one_document(
    doc_item,
    ref_item,
    I=I_small,
    method="tfidf",
    beta=0.5,
    p=3
)

print(result["doc_id"], result["category"])
print("Precision:", result["precision"])
print("Recall:", result["recall"])
print("F0.5:", result["fbeta"])
print("Average Precision:", result["average_precision"])

business_001 business
Precision: 0.6666666666666666
Recall: 0.2857142857142857
F0.5: 0.5263157894736842
Average Precision: 0.5911848072562359


#### E.4 Evaluate multiple

In [59]:
def evaluate_collection(docs_subset, ref_lookup, I, method="tfidf", beta=0.5, p=3, args=None):
    
    results = []
    
    for doc_item in docs_subset:
        
        doc_id = doc_item["id"]
        
        if doc_id not in ref_lookup:
            continue
            
        ref_item = ref_lookup[doc_id]
        
        result = evaluate_one_document(
            doc_item,
            ref_item,
            I=I,
            method=method,
            beta=beta,
            p=p,
            args=args
        )
        
        results.append(result)
        
    return results

In [61]:
import numpy as np

def summarize_evaluation_results(results):

    precision = np.mean([r["precision"] for r in results])
    recall = np.mean([r["recall"] for r in results])
    fbeta = np.mean([r["fbeta"] for r in results])
    ap = np.mean([r["average_precision"] for r in results])

    return {
        "Precision": precision,
        "Recall": recall,
        "F0.5": fbeta,
        "MAP": ap,
        "Docs evaluated": len(results)
    }

In [62]:
eval_docs = docs[:100]
eval_ref_lookup = {r["id"]: r for r in refs if r["id"] in {d["id"] for d in eval_docs}}

methods_to_test = [
    ("tfidf", {}),
    ("bm25", {}),
    ("embedding", {"dense_model": "all-MiniLM-L6-v2"}),
    ("embedding", {"dense_model": "multi-qa-MiniLM-L6-cos-v1"}),
]

for method, extra_args in methods_to_test:
    print("\nMETHOD:", method, extra_args)

    results = evaluate_collection(
        eval_docs,
        eval_ref_lookup,
        I=I_small,
        beta=0.5,
        p=3,
        method=method,
        args=extra_args
    )

    summary = summarize_evaluation_results(results)
    print(summary)


METHOD: tfidf {}
{'Precision': 0.7600000000000001, 'Recall': 0.4178142135642136, 'F0.5': 0.6390996162644433, 'MAP': 0.746319528211065, 'Docs evaluated': 100}

METHOD: bm25 {}
{'Precision': 0.59, 'Recall': 0.32209992784992786, 'F0.5': 0.49510089246502287, 'MAP': 0.6167431181947076, 'Docs evaluated': 100}

METHOD: embedding {'dense_model': 'all-MiniLM-L6-v2'}
{'Precision': 0.6900000000000002, 'Recall': 0.3794613997113997, 'F0.5': 0.5804409498646609, 'MAP': 0.6802782486255577, 'Docs evaluated': 100}

METHOD: embedding {'dense_model': 'multi-qa-MiniLM-L6-cos-v1'}
{'Precision': 0.7000000000000002, 'Recall': 0.38735028860028864, 'F0.5': 0.5880486710908032, 'MAP': 0.693147784780139, 'Docs evaluated': 100}


#### E.5 Category-level evaluation

In [63]:
from collections import defaultdict

def evaluate_by_category(docs_subset, ref_lookup, I, method="tfidf", beta=0.5, p=3, args=None):
    grouped = defaultdict(list)
    for d in docs_subset:
        grouped[d["category"]].append(d)

    category_stats = {}

    for category, cat_docs in grouped.items():
        results = evaluate_collection(
            cat_docs,
            ref_lookup,
            I=I,
            method=method,
            beta=beta,
            p=p,
            args=args
        )
        category_stats[category] = summarize_evaluation_results(results)

    return category_stats

In [64]:
cat_tfidf = evaluate_by_category(
    eval_docs,
    eval_ref_lookup,
    I=I_small,
    method="tfidf",
    beta=0.5,
    p=3
)

cat_dense = evaluate_by_category(
    eval_docs,
    eval_ref_lookup,
    I=I_small,
    method="embedding",
    beta=0.5,
    p=3,
    args={"dense_model": "multi-qa-MiniLM-L6-cos-v1"}
)

print("TF-IDF by category")
for cat, stats in cat_tfidf.items():
    print(cat, stats)

print("\nDense by category")
for cat, stats in cat_dense.items():
    print(cat, stats)

TF-IDF by category
business {'Precision': 0.7600000000000001, 'Recall': 0.4178142135642136, 'F0.5': 0.6390996162644433, 'MAP': 0.746319528211065, 'Docs evaluated': 100}

Dense by category
business {'Precision': 0.7000000000000002, 'Recall': 0.38735028860028864, 'F0.5': 0.5880486710908032, 'MAP': 0.693147784780139, 'Docs evaluated': 100}


#### E.6 MMR evaluation

In [65]:
def evaluate_collection_with_custom_args(docs_subset, ref_lookup, I, beta=0.5, p=3, args=None):
    results = []

    for doc_item in docs_subset:
        doc_id = doc_item["id"]
        if doc_id not in ref_lookup:
            continue

        ref_item = ref_lookup[doc_id]

        result = evaluate_one_document(
            doc_item,
            ref_item,
            I=I,
            method=args.get("method", "tfidf"),
            beta=beta,
            p=p,
            args=args
        )
        results.append(result)

    return results

In [74]:
results_dense = evaluate_collection_with_custom_args(
    eval_docs,
    eval_ref_lookup,
    I=I_small,
    beta=0.5,
    p=3,
    args={
        "method": "embedding",
        "dense_model": "multi-qa-MiniLM-L6-cos-v1",
        "use_mmr": False
    }
)

results_dense_mmr = evaluate_collection_with_custom_args(
    eval_docs,
    eval_ref_lookup,
    I=I_small,
    beta=0.5,
    p=3,
    args={
        "method": "embedding",
        "dense_model": "multi-qa-MiniLM-L6-cos-v1",
        "use_mmr": True,
        "lambda_param": 0.5
    }
)

print("Dense without MMR")
print(summarize_evaluation_results(results_dense))

print("\nDense with MMR")
print(summarize_evaluation_results(results_dense_mmr))

Dense without MMR
{'Precision': 0.7000000000000002, 'Recall': 0.38735028860028864, 'F0.5': 0.5880486710908032, 'MAP': 0.693147784780139, 'Docs evaluated': 100}

Dense with MMR
{'Precision': 0.58, 'Recall': 0.3262940115440116, 'F0.5': 0.4931522026846936, 'MAP': 0.693147784780139, 'Docs evaluated': 100}


#### E.7 RRF evaluation

In [75]:
def evaluate_one_document_rrf(doc_item, ref_item, I, beta=0.5, p=3, args=None, threshold=0.6):

    if args is None:
        args = {}

    doc_text = doc_item["text"]
    ref_text = ref_item["summary"]

    summary_output = rrf_summarization(
        doc_text,
        p=p,
        I=I,
        args=args
    )

    predicted_sentences = extract_selected_sentences(doc_text, summary_output)
    reference_sentences = get_reference_sentences(ref_text)

    prf = precision_recall_fbeta(
        predicted_sentences=predicted_sentences,
        reference_sentences=reference_sentences,
        beta=beta,
        threshold=threshold
    )

    return {
        "doc_id": doc_item["id"],
        "category": doc_item["category"],
        "precision": prf["precision"],
        "recall": prf["recall"],
        "fbeta": prf["fbeta"],
        "average_precision": 0.0
    }

In [76]:
def evaluate_collection_rrf(docs_subset, ref_lookup, I, beta=0.5, p=3, args=None):

    results = []

    for doc_item in docs_subset:

        doc_id = doc_item["id"]

        if doc_id not in ref_lookup:
            continue

        ref_item = ref_lookup[doc_id]

        res = evaluate_one_document_rrf(
            doc_item,
            ref_item,
            I=I,
            beta=beta,
            p=p,
            args=args
        )

        results.append(res)

    return results

In [77]:
results_rrf = evaluate_collection_rrf(
    eval_docs,
    eval_ref_lookup,
    I=I_small,
    beta=0.5,
    p=3,
    args={
        "sparse_method": "bm25",
        "dense_model": "multi-qa-MiniLM-L6-cos-v1",
        "mu": 5
    }
)

print("RRF (BM25 + multi-qa)")
print(summarize_evaluation_results(results_rrf))

RRF (BM25 + multi-qa)
{'Precision': 0.6466666666666666, 'Recall': 0.3579848484848485, 'F0.5': 0.5439329814286907, 'MAP': 0.0, 'Docs evaluated': 100}


#### E.8 Abstractive evaluation

In [71]:
_eval_encoder_cache = {}

def get_eval_encoder(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    if model_name not in _eval_encoder_cache:
        _eval_encoder_cache[model_name] = SentenceTransformer(model_name)
    return _eval_encoder_cache[model_name]

In [72]:
def evaluate_abstractive_summary(doc_item, ref_item, I, model_name, prompt_type,
                                 eval_encoder_name="sentence-transformers/all-MiniLM-L6-v2"):
    generated = abstractive_summarization(
        doc_item["text"],
        p=3,
        I=I,
        args={
            "model_name": model_name,
            "prompt_type": prompt_type,
            "max_new_tokens": 60
        }
    )

    reference = ref_item["summary"]

    encoder = get_eval_encoder(eval_encoder_name)
    emb = encoder.encode([generated, reference], convert_to_numpy=True)

    sim = cosine_similarity(
        emb[0].reshape(1, -1),
        emb[1].reshape(1, -1)
    )[0][0]

    return {
        "doc_id": doc_item["id"],
        "category": doc_item["category"],
        "model_name": model_name,
        "prompt_type": prompt_type,
        "generated_summary": generated,
        "reference_summary": reference,
        "embedding_similarity": float(sim)
    }

In [73]:
abstractive_test_docs = docs[:5]

for doc_item in abstractive_test_docs:

    ref_item = ref_lookup[doc_item["id"]]

    res = evaluate_abstractive_summary(
        doc_item,
        ref_item,
        I=I_small,
        model_name="gpt2",
        prompt_type="focused"
    )

    print("\nDOC:", res["doc_id"])
    print("Similarity:", res["embedding_similarity"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1605.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



DOC: business_001
Similarity: 0.09653250873088837

DOC: business_002
Similarity: 0.4981856644153595

DOC: business_003
Similarity: 0.20177817344665527

DOC: business_004
Similarity: 0.4223257303237915

DOC: business_005
Similarity: 0.3347502648830414


In [70]:
def evaluation(Sset=None, Rset=None, args=None):
    """
    Project wrapper for evaluation.
    This notebook mainly uses helper functions for extractive and abstractive evaluation.
    """
    if args is None:
        args = {}

    mode = args.get("mode", "extractive")

    if mode == "extractive":
        return {
            "message": "Use evaluate_one_document, evaluate_collection, evaluate_by_category, evaluate_collection_with_custom_args, or evaluate_collection_rrf for extractive evaluation."
        }

    elif mode == "abstractive":
        return {
            "message": "Use evaluate_abstractive_summary for embedding-based abstractive evaluation."
        }

    else:
        raise ValueError("Unknown evaluation mode")

<H3>Part II: questions materials (optional)</H3>

**(1)** Keyword extraction using classic *vs* generative stances

In [56]:
#code, statistics and/or charts here

**(2)** Summarization performance for the overall and category-conditional corpora.

In [57]:
#code, statistics and/or charts here

**...** (additional questions with empirical results)

<H3>END</H3>